# Edmonton Rent Model

This notebook inspects the final rent model without duplicating the training logic. The shared `train_model()` function loads the model-ready data, compares the baseline with linear regression, evaluates the selected model on the held-out test years, and refits it on all available data for production use.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from models.train_model import MODEL_DATA_PATH, train_model

results = train_model(MODEL_DATA_PATH)

## Final Test Results

The test period contains the four years from 2022 through 2025. These observations were not used to select the winning model.

In [2]:
model_name = str(results["selected_model"]).replace("_", " ").title()
print(f"Selected model: {model_name}")
print(f"MAE: {results['test_mae']:.2f}")
print(f"RMSE: {results['test_rmse']:.2f}")
print(f"R²: {results['test_r2']:.3f}")
print("\nCoefficients (production model, refit on all available data):")
if results["coefficients"] is None:
    print("Not applicable: the baseline model has no learned coefficients.")
    print("\nIntercept: Not applicable")
else:
    for feature, coefficient in results["coefficients"].items():
        print(f"{feature}: {coefficient:.4f}")
    print(f"\nIntercept: {results['intercept']:.4f}")

Selected model: Linear Regression
MAE: 62.06
RMSE: 65.80
R²: 0.669

Coefficients (production model, refit on all available data):
rent_lag1: 1.0412
vacancy_lag1: -13.8905

Intercept: 46.2515


## Interpretation

- **MAE** is the average absolute difference between predicted and observed monthly rent, expressed in dollars.
- **RMSE** is also measured in dollars but gives greater weight to larger prediction errors.
- **R²** describes the share of variation in the four test observations explained by the predictions. It should be interpreted cautiously because the test set is small.
- The coefficients below belong to the production model, which was refitted using all available years after model selection and final testing.

In [3]:
coefficients = results["coefficients"]
if coefficients is not None:
    print(
        "Holding last year's vacancy rate constant, a $1 increase in last year's "
        f"rent is associated with a ${coefficients['rent_lag1']:.2f} increase in "
        "this year's predicted rent."
    )
    vacancy_direction = "increase" if coefficients["vacancy_lag1"] >= 0 else "decrease"
    print(
        "Holding last year's rent constant, a one-percentage-point increase in "
        "last year's vacancy rate is associated with a "
        f"${abs(coefficients['vacancy_lag1']):.2f} {vacancy_direction} in this "
        "year's predicted rent."
    )
    print(
        "The intercept is the fitted rent when both lagged inputs are zero, so it "
        "is mainly a mathematical model component rather than a realistic market estimate."
    )
else:
    print("The selected baseline predicts this year's rent from last year's rent and has no learned parameters.")

Holding last year's vacancy rate constant, a $1 increase in last year's rent is associated with a $1.04 increase in this year's predicted rent.
Holding last year's rent constant, a one-percentage-point increase in last year's vacancy rate is associated with a $13.89 decrease in this year's predicted rent.
The intercept is the fitted rent when both lagged inputs are zero, so it is mainly a mathematical model component rather than a realistic market estimate.


## Important limitation

These coefficients describe associations in this small historical dataset; they do not establish that changes in vacancy caused changes in rent. With only 25 model-ready annual observations and four test years, the metrics and coefficients should be treated as a simple forecasting benchmark rather than strong evidence about the rental market.